[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milioe/casos-ia-ibero-diplomado/blob/main/modulo_4/09_Pretrained_Embedding.ipynb)


# Embeddings preentrenados — buscar en una lista con una pregunta

En **07** y **08** entrenaste Word2Vec con *tu* texto. Aquí usamos modelos que **ya vienen entrenados**: mandas frases por una API y recibes vectores listos.

**Ejercicio de hoy:** tenemos una **lista de 10 párrafos** — como entradas de un FAQ de telefonía (facturación, roaming, soporte, etc.). Algunos están en **español** y otros en **inglés** a propósito. Llega una **pregunta** del usuario. ¿Cuál párrafo es el más parecido en *significado*?

No es búsqueda por palabra exacta (como Ctrl+F). El embedding entiende que *"¿por qué no tengo señal en el extranjero?"* puede acercarse a un párrafo en inglés sobre *roaming* aunque no compartan las mismas palabras.

Pasos:

1. Conectar con la **Gemini API**.
2. Ver qué devuelve el modelo para **una frase**.
3. Revisar la **lista de 10 párrafos** + escribir una **pregunta**.
4. Calcular similitud y decir **cuál gana**.
5. Repetir el mismo ejercicio con **otro proveedor** — NVIDIA Nemotron-3-Embed.

**Este notebook está pensado para correr en Google Colab** (usa el panel de *Secrets* para las llaves), no en local.

## Setup — instalar librerías

- `google-genai` — cliente de la Gemini API.
- `numpy` — comparar vectores.
- `requests` — llamar la API de NVIDIA más adelante (sección 8).

In [ ]:
%pip install -q google-genai numpy requests

In [ ]:
import os
import textwrap

import numpy as np
import requests
from google import genai


## 1 — ¿Qué cambia respecto a Word2Vec?

| | Word2Vec (notebooks 07–08) | Embedding preentrenado (este cuaderno) |
|---|---|---|
| **Quién entrena** | Tú, con gensim/Keras | Google (millones de textos) |
| **Unidad típica** | Una **palabra** | Una **frase** o párrafo completo |
| **Dónde corre** | Tu laptop / Colab | Servidor de Google (API) |
| **Costo** | Gratis (solo tu tiempo) | Tier gratuito con límites; luego de pago |
| **Privacidad** | Todo local | El texto sale a internet |

La idea del vector es la misma: una lista de números que resume el **significado**. Si dos frases dicen cosas parecidas, sus vectores deberían quedar **cerca** (similitud coseno alta).


## 2 — Clave API (Google AI Studio, desde Colab)

1. Entra a [Google AI Studio → API keys](https://aistudio.google.com/apikey) y crea una clave.
2. En Colab, abre el panel de **Secrets** (ícono de llave 🔑 en la barra lateral izquierda) y guárdala con el nombre `GOOGLE_API_KEY`. Actívala para este notebook.

Usaremos el modelo **`gemini-embedding-001`**: solo texto, multilingüe (incluye español). Documentación: [Embeddings | Gemini API](https://ai.google.dev/gemini-api/docs/embeddings).

In [ ]:
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
API_KEY = os.environ["GOOGLE_API_KEY"].strip()

cliente = genai.Client(api_key=API_KEY)
MODELO_EMBEDDING = "gemini-embedding-001"

print(f"Cliente listo. Modelo: {MODELO_EMBEDDING}")

## 3 — Una frase muy sencilla

Mandamos **un solo texto** al método `embed_content`. La API devuelve un objeto con la lista de números (el embedding).

Piensa en el vector como una **huella numérica** de la frase: no es legible para humanos, pero sirve para comparar textos.


In [ ]:
frase = "El gato duerme en el sofá."

respuesta = cliente.models.embed_content(
    model=MODELO_EMBEDDING,
    contents=frase,
)

# El SDK devuelve una lista de embeddings; con un solo texto, tomamos el primero.
vector = np.array(respuesta.embeddings[0].values, dtype=np.float32)

print(f"Frase: {frase}")
print(f"Dimensiones del vector: {vector.shape[0]}")
print(f"Primeros 8 valores: {vector[:8]}")
print(f"Norma L2 (longitud del vector): {np.linalg.norm(vector):.4f}")

El modelo devuelve una lista larga de números (3072 por defecto). No hace falta leer cada cifra: lo usaremos para **comparar** textos entre sí.


## 4 — Funciones para comparar textos

Dos piezas: convertir texto → vector, y medir qué tan parecidos son dos vectores (**similitud coseno**: 1 = muy parecido, 0 = nada que ver).


In [ ]:
def embed_textos(textos):
    """Devuelve matriz (n_textos, n_dims) con los embeddings de Gemini."""
    if isinstance(textos, str):
        textos = [textos]
    resp = cliente.models.embed_content(
        model=MODELO_EMBEDDING,
        contents=textos,
    )
    return np.array([e.values for e in resp.embeddings], dtype=np.float32)


def similitud_coseno(a, b):
    """1.0 = misma dirección; 0 = ortogonales; valores negativos = opuestos."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

## 5 — Lista de FAQ (telefonía) + una pregunta

Simulamos preguntas frecuentes de una **compañía de telefonía**: facturación, planes, roaming, soporte técnico, portabilidad, etc.

Cada elemento es un **párrafo de dos líneas** (como un artículo del reglamento o del centro de ayuda). Algunos están en **inglés** para ver si el modelo conecta significado aunque la pregunta venga en español.

Cambia `pregunta` en la celda siguiente y vuelve a ejecutar desde ahí.

In [ ]:
FAQ = [
    # 1 — Facturación (ES)
    "Si tu recibo mensual incluye cargos que no reconoces, revisa primero el desglose en la app Mi Cuenta "
    "y confirma si hubo consumo de datos fuera de plan o servicios de terceros activados por SMS. "
    "Para disputar un cargo, abre un ticket antes del día 10 posterior a la emisión de la factura.",

    # 2 — Roaming (EN)
    "If your mobile line shows No Service after traveling abroad, verify that data roaming is enabled "
    "in your device settings and that your plan includes international coverage. "
    "Contact support within 48 hours to restore your line without penalty if the issue persists.",

    # 3 — Cambio de plan (ES)
    "Para subir o bajar tu plan de datos, entra a Mi Cuenta con tu número registrado y elige la opción "
    "Cambiar plan en el menú de servicios. Los cambios aplican al inicio del siguiente ciclo de facturación, "
    "salvo que contrates un paquete adicional de datos, que se activa de inmediato.",

    # 4 — Sin señal / red (EN)
    "When calls drop frequently or you see only one bar indoors, restart your phone and confirm that "
    "VoLTE and automatic network selection are turned on. If the problem continues in the same location "
    "for more than 72 hours, request a network quality review from technical support.",

    # 5 — Portabilidad (ES)
    "La portabilidad numérica permite conservar tu número al cambiarte de compañía sin costo, siempre "
    "que no tengas adeudos pendientes ni contrato de permanencia vigente. Solicítala en la tienda "
    "de la nueva operadora con una identificación oficial; el trámite tarda entre 24 y 72 horas hábiles.",

    # 6 — Formas de pago (ES)
    "Puedes pagar tu factura en OXXO, transferencia SPEI, tarjeta de crédito o débito desde la app, "
    "y en ventanilla bancaria con tu referencia de 16 dígitos. El pago se refleja en un máximo de "
    "48 horas; después de esa fecha pueden aplicarse recargos por mora según tu contrato.",

    # 7 — Garantía del equipo (EN)
    "Devices purchased from our stores include a 12-month warranty against manufacturing defects, "
    "but not damage from liquids, falls, or unauthorized repairs. To start a warranty claim, "
    "bring the phone, the original invoice, and the IMEI number to any authorized service center.",

    # 8 — Fraude / SIM swap (ES)
    "Si recibes llamadas extrañas pidiendo tu NIP o códigos de verificación, no los compartas: "
    "puede tratarse de un intento de fraude o clonación de SIM. Reporta el incidente de inmediato "
    "al *611 o al chat de seguridad para bloquear tu línea y emitir una SIM nueva.",

    # 9 — Internet en casa (ES)
    "La instalación de fibra óptica incluye visita técnica, módem y configuración de la red Wi-Fi "
    "en tu domicilio; el técnico verificará que haya cobertura en tu colonia antes de perforar. "
    "Si ya tienes servicio y la velocidad es menor a la contratada, realiza una prueba por cable LAN.",

    # 10 — Horarios de atención (EN)
    "Customer service by phone is available Monday through Saturday from 8:00 a.m. to 8:00 p.m., "
    "and Sundays from 9:00 a.m. to 2:00 p.m. local time. For faster help, use the in-app chat "
    "or visit a retail store with your account number and a valid ID.",
]

print(f"Entradas en la lista: {len(FAQ)}")
for i, texto in enumerate(FAQ, start=1):
    print(f"\n--- [{i}] ---")
    print(textwrap.fill(texto, width=88))

In [ ]:
pregunta = "¿Por qué no tengo señal desde que aterricé en otro país?"

In [ ]:
print("Pregunta del usuario:")
print(f"  → {pregunta}")

## 6 — ¿Cuál párrafo de la lista gana?

Una llamada a la API: embedding de la pregunta + embedding de cada párrafo. **Mayor similitud coseno = más parecido en significado.**

In [ ]:
def imprimir_parrafo(texto, indent="   ", ancho=88):
    """Envuelve el texto para que no se corte feo en pantalla."""
    return textwrap.fill(texto, width=ancho, initial_indent=indent, subsequent_indent=indent)


def buscar_en_lista(pregunta, textos):
    """Devuelve lista ordenada: (índice, párrafo, similitud) de mayor a menor."""
    todos = [pregunta] + list(textos)
    vectores = embed_textos(todos)
    vec_pregunta = vectores[0]
    vec_entradas = vectores[1:]

    resultados = []
    for i, (texto, vec_entrada) in enumerate(zip(textos, vec_entradas)):
        score = similitud_coseno(vec_pregunta, vec_entrada)
        resultados.append((i + 1, texto, score))

    resultados.sort(key=lambda x: x[2], reverse=True)
    return resultados


ranking = buscar_en_lista(pregunta, FAQ)

print(f"Pregunta: {pregunta}\n")
print("Ranking (de más parecido a menos):")
print("=" * 88)

for pos, (num, texto, score) in enumerate(ranking, start=1):
    print(f"\n{pos:2d}. [#{num}]  similitud = {score:.4f}")
    print(imprimir_parrafo(texto))

ganador = ranking[0]
print("\n" + "=" * 88)
print(f"MEJOR COINCIDENCIA: #{ganador[0]}  (similitud = {ganador[2]:.4f})")
print("=" * 88)
print(imprimir_parrafo(ganador[1], indent=""))

## 7 — Prueba con otra pregunta

Cambia `pregunta` en la celda de arriba y vuelve a correr las celdas **5** y **6**. Ideas:

- `"Me cobraron algo raro en el recibo del mes"` → debería ir a **#1** (facturación)
- `"¿Cómo me cambio de compañía y me quedo con mi número?"` → **#5** (portabilidad)
- `"My phone screen broke after I dropped it, is it covered?"` → **#7** (garantía, en inglés)
- `"¿Hasta qué hora puedo marcar al call center?"` → **#10** (horarios, en inglés)

**Cierre.** Misma lógica que un chatbot de soporte: la pregunta entra, se compara contra una lista de respuestas guardadas, gana la más parecida. El modelo multilingüe ayuda cuando la pregunta y el párrafo no están en el mismo idioma.

## 8 — Otro proveedor: NVIDIA (Nemotron-3-Embed)

Gemini no es la única API de embeddings. **NVIDIA** ofrece `nemotron-3-embed-1b` gratis (con límites) en [build.nvidia.com](https://build.nvidia.com/nvidia/nemotron-3-embed-1b) — un modelo de 1B de parámetros pensado para búsqueda semántica y RAG, con vectores de 2048 dimensiones.

La diferencia con Gemini: aquí hay que decir explícitamente si el texto es una **pregunta** (`input_type="query"`) o un **documento que vas a indexar** (`input_type="passage"`). Usarlo al revés baja mucho la calidad de la búsqueda.

### Clave API (Colab)

1. Entra a [build.nvidia.com/nvidia/nemotron-3-embed-1b](https://build.nvidia.com/nvidia/nemotron-3-embed-1b) y da clic en **Get API Key**.
2. Guárdala en el panel de **Secrets** de Colab con el nombre `NVIDIA_API_KEY`.

In [ ]:
from google.colab import userdata

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")
NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"].strip()

print("Llave de NVIDIA lista.")

### Llamar la API

`https://integrate.api.nvidia.com/v1/embeddings` es compatible con el formato de OpenAI: mandas `input` (uno o varios textos) y regresa un vector por texto. Estos son los parámetros que también ves en el playground de NVIDIA ([Try](https://build.nvidia.com/nvidia/nemotron-3-embed-1b)) — todos son obligatorios:

| Parámetro | Valores | Para qué sirve |
|---|---|---|
| `input` | texto o lista de textos | Lo que quieres embeber. Máximo 4096 tokens por texto. |
| `model` | `nvidia/nemotron-3-embed-1b` | Qué modelo usar (aquí solo hay uno). |
| `input_type` | `query` / `passage` | **El único parámetro que Gemini no tiene.** `query` = es una pregunta de búsqueda. `passage` = es un documento que vas a indexar. Usarlo al revés no truena, pero baja mucho la calidad del ranking. |
| `encoding_format` | `float` / `base64` | En qué formato regresa el vector. Usamos `float` (una lista de números) porque es la que se puede meter directo a NumPy; `base64` pesa menos en la respuesta pero hay que decodificarla. |
| `truncate` | `NONE` / `START` / `END` | Qué hacer si el texto es más largo que el máximo del modelo. `END` descarta el final del texto; `START` descarta el principio; `NONE` regresa un error en vez de recortar. Usamos `END` para no truncar el inicio de la pregunta o del párrafo. |

**`truncate`, con más detalle:** el modelo tiene un límite fijo de entrada (4096 tokens). Si tu texto no cabe, algo se recorta:

- `END` → se queda con el **inicio** del texto, corta el final. Es lo que usamos aquí: en una pregunta o un párrafo, la idea principal suele estar al principio.
- `START` → se queda con el **final**, corta el inicio. Riesgoso: puedes perder justo la parte que dice de qué trata el texto.
- `NONE` → no recorta nada; si no cabe, la API regresa un **error** en vez de adivinar qué cortar.

Con las preguntas y párrafos cortos de este notebook nunca se llega ni cerca del límite, así que aquí `truncate` no hace nada visible — importa cuando mandas documentos largos, como páginas completas de un PDF (notebook 11).

**Nota:** `input_type` es obligatorio porque Nemotron-3-Embed es un modelo *asimétrico* — entrena por separado cómo representar preguntas y cómo representar documentos, así que necesita saber cuál de los dos le estás mandando. Gemini es *simétrico*: usa la misma representación para ambos, por eso no le pedimos nada parecido en la sección 3.

In [ ]:
NVIDIA_EMBEDDINGS_URL = "https://integrate.api.nvidia.com/v1/embeddings"
MODELO_NVIDIA = "nvidia/nemotron-3-embed-1b"


def embed_textos_nvidia(textos, input_type="query"):
    """Devuelve matriz (n_textos, n_dims) con los embeddings de NVIDIA.

    input_type="query"   -> el texto es una pregunta de búsqueda.
    input_type="passage" -> el texto es un documento que se va a indexar.
    """
    if isinstance(textos, str):
        textos = [textos]

    resp = requests.post(
        NVIDIA_EMBEDDINGS_URL,
        headers={
            "Authorization": f"Bearer {NVIDIA_API_KEY}",
            "Accept": "application/json",
        },
        json={
            "input": textos,
            "model": MODELO_NVIDIA,
            "input_type": input_type,
            "encoding_format": "float",
            "truncate": "END",
        },
        timeout=60,
    )
    resp.raise_for_status()
    datos = resp.json()["data"]
    return np.array([d["embedding"] for d in datos], dtype=np.float32)


vector_nvidia = embed_textos_nvidia("El gato duerme en el sofá.", input_type="passage")[0]
print(f"Dimensiones del vector: {vector_nvidia.shape[0]}")
print(f"Primeros 8 valores: {vector_nvidia[:8]}")

### Mismo FAQ, mismo `pregunta`, otro proveedor

Reusamos la lista `FAQ` y la variable `pregunta` de la sección 5. Los párrafos del FAQ se embeben como `passage` (se van a indexar); la pregunta se embebe como `query`.

In [ ]:
vectores_faq_nvidia = embed_textos_nvidia(FAQ, input_type="passage")
vector_pregunta_nvidia = embed_textos_nvidia(pregunta, input_type="query")[0]

resultados_nvidia = [
    (i + 1, texto, similitud_coseno(vector_pregunta_nvidia, vec))
    for i, (texto, vec) in enumerate(zip(FAQ, vectores_faq_nvidia))
]
resultados_nvidia.sort(key=lambda x: x[2], reverse=True)

print(f"Pregunta: {pregunta}\n")
print("Ranking con NVIDIA (de más parecido a menos):")
print("=" * 88)
for pos, (num, texto, score) in enumerate(resultados_nvidia, start=1):
    print(f"\n{pos:2d}. [#{num}]  similitud = {score:.4f}")
    print(imprimir_parrafo(texto))

ganador_nvidia = resultados_nvidia[0]
print("\n" + "=" * 88)
print(f"MEJOR COINCIDENCIA (NVIDIA): #{ganador_nvidia[0]}  (similitud = {ganador_nvidia[2]:.4f})")

**Cierre.** Compara el ranking de esta sección con el de la sección 6 (Gemini) — con esta pregunta, normalmente coinciden en el ganador. Cambiar de proveedor no cambia el patrón: embeber texto → similitud coseno → ranking. Lo que cambia es la llave, el endpoint y, aquí, el detalle extra de `input_type`.